In [1]:
#Goal: Need to create a model that can predict sentiment of Happy or Sad in Yelp Data.

In [2]:
import pandas as pd
import numpy as np

In [17]:
# File is processed for Happy and Sad in Sentiment column withint Excel, only column, feature and label left.
data = pd.read_csv('Yelp.csv',sep=',',names=['Review'])

In [18]:
data.head()

,Review
0,My wife took me here on my birthday for breakf...
1,I have no idea why some people give bad review...
2,love the gyro plate. Rice is so good and I als...
3,"Rosie, Dakota, and I LOVE Chaparral Dog Park!!..."
4,General Manager Scott Petello is a good egg!!!...


In [19]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Review  10000 non-null  object
dtypes: object(1)
memory usage: 78.2+ KB


In [6]:
!pip install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 9.3 MB/s eta 0:00:00


In [20]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Initialize VADER Sentiment Analyzer
analyzer = SentimentIntensityAnalyzer()

# Define a function to classify sentiment based on compound score
def classify_sentiment(text):
    # Run VADER sentiment analysis on the text
    score = analyzer.polarity_scores(text)['compound']

    # Classify as 'happy', 'sad', or 'neutral' based on the compound score
    if score >= 0.00:
        return 'happy'
    else:
        return 'sad'

In [21]:
# Since the name of the column is 'Review'
# Create a new 'sentiment' column in the dataset
data['label'] = data['Review'].apply(classify_sentiment)

In [22]:
# Save the new dataset with the added sentiment column
data.to_csv('yelp_with_vader_binary_sentiment.csv', index=False)

In [24]:
data.label.value_counts()

,count
label,
happy,9064
sad,936


In [25]:
data.head()

,Review,label
0,My wife took me here on my birthday for breakf...,happy
1,I have no idea why some people give bad review...,happy
2,love the gyro plate. Rice is so good and I als...,happy
3,"Rosie, Dakota, and I LOVE Chaparral Dog Park!!...",happy
4,General Manager Scott Petello is a good egg!!!...,happy


In [26]:
# No missing value from info so no need to removal of data.

In [27]:
#Binarize labels
data['label']=data['label'].map({'sad':0, 'happy':1})
data.head()

,Review,label
0,My wife took me here on my birthday for breakf...,1
1,I have no idea why some people give bad review...,1
2,love the gyro plate. Rice is so good and I als...,1
3,"Rosie, Dakota, and I LOVE Chaparral Dog Park!!...",1
4,General Manager Scott Petello is a good egg!!!...,1


In [28]:
#Seperate data as features and label
features = data.Review.values
label = data.label.values

In [29]:
#Train test split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(features,
                                                 label,
                                                 test_size=0.2,
                                                 random_state = 1)

In [30]:
# For Tokenization
from tensorflow.keras.preprocessing.text import Tokenizer

#Decide the Vocabulary Word Frequency Size

vocabFreqWordSize = None

# If the word 'spiderman' exists a minimum of num_words times in the corpus,
# then it will be added in the vocab dictionary

#Convert the sentences into sequence of words

tokenizer = Tokenizer(num_words=vocabFreqWordSize)

#Fit the training data

tokenizer.fit_on_texts(X_train)


In [31]:
#Lets create the sequence objects

seq_train = tokenizer.texts_to_sequences(X_train)
seq_test = tokenizer.texts_to_sequences(X_test)

In [32]:
X_train[0]

'Great girls night out place.  Wine selection was extensive and we DEVOURED the Bruschetta!  Try to get there for the Happy Hour prices or you will be paying a minimum of $9 per glass of wine.  \nOne of my new fave places :)'

In [33]:
len(seq_train[3])

59

In [34]:
len(seq_train)

8000

In [35]:
#Lets pad the sequence data
#This creates a matrix where the
# rows = Documents
# Cols = Time Steps (For our understanding its the vocab with padding)

from tensorflow.keras.preprocessing.sequence import pad_sequences

trainData = pad_sequences(seq_train)
T = trainData.shape[1]
T

953

In [36]:
testData = pad_sequences(seq_test , maxlen=T)
testData.shape

(2000, 953)

In [37]:
#Build the Model
# Preferred model is CNN due to data size and avoiding memory overflow issue
# and without compromising the quality of the data.

In [38]:
#Get the Vocab Size of the Tokenizer
V = len(tokenizer.word_index)

In [80]:
#CNN for Text data -- Conv1D



from tensorflow.keras.layers import Dense,Input,Conv1D,Embedding,MaxPooling1D,GlobalMaxPooling1D
from tensorflow.keras.models import Model

# 1. Create/Convert Seq data into Word Embedding / Embedded Data

#Input layer -  It takes in sequence of Integers i.e. T  (time step size ////// vocab size)

i = Input(shape=(T,))

#Create WordEmbedding ---- This layer will take sequence of integers and return sequence of word vectors
# Ideally input size is No of Docs X Time Steps
# When creating word embedding the dimension of matrix will be (Total_Vocabulary_Size + 1,Embedding Dim)
# Reason for + 1 is indexing of embedding starts from 1 and not 0


#You can decide the Embedding Dimensionality -- Hyperparameter
#Prashant Recommend ----> 10 to 100
D = 20

x = Embedding(V + 1,D)(i)

#First Convolution Layer

x = Conv1D(32,3,activation='relu')(x)
x = MaxPooling1D(3)(x)

#Second Convolution Layer

x = Conv1D(64,3,activation='relu')(x)
x = MaxPooling1D(3)(x)

#Third Convolution Layer

x = Conv1D(128,3,activation='relu')(x)
#x = Flatten()(x)
x = GlobalMaxPooling1D()(x)

#Dense Layer -- Output Layer

x = Dense(1,activation='sigmoid')(x)


model = Model(i,x)

In [81]:
# Summary of the model
model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)           │ (None, 953)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ embedding_7 (Embedding)              │ (None, 953, 20)             │         556,760 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_15 (Conv1D)                   │ (None, 951, 32)             │           1,952 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_10 (MaxPooling1D)      │ (None, 317, 32)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_16 (Conv1D)                   │ (None, 315, 64)             │           6,208 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_11 (MaxPooling1D)      │ (None, 105, 64)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_17 (Conv1D)                   │ (None, 103, 128)            │          24,704 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d_5               │ (None, 128)                 │               0 │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_10 (Dense)                     │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 589,753 (2.25 MB)

 Trainable params: 589,753 (2.25 MB)

 Non-trainable params: 0 (0.00 B)

In [82]:
#Custom Callback for early stopping
# Import necessary libraries
import tensorflow as tf
class MyThresholdCallBack(tf.keras.callbacks.Callback):
    def __init__(self,cl):
        super(MyThresholdCallBack, self).__init__()
        self.cl = cl

    def on_epoch_end(self, epoch, logs=None):
        test_score = logs["val_accuracy"]
        train_score = logs["accuracy"]

        if test_score > train_score and test_score > self.cl:
        #if test_score > self.cl:
            self.model.stop_training = True

In [83]:
# Create the custom callback instance
myR2ScoreMonitor = MyThresholdCallBack(cl=0.91)

In [84]:
#Compile
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [85]:
#Train
history = model.fit(trainData,y_train, epochs=50, validation_data=(testData,y_test),callbacks=[myR2ScoreMonitor])

Epoch 1/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.8831 - loss: 0.3614 - val_accuracy: 0.9085 - val_loss: 0.2455
Epoch 2/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9135 - loss: 0.2048 - val_accuracy: 0.9140 - val_loss: 0.2130
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9559 - loss: 0.1125 - val_accuracy: 0.9195 - val_loss: 0.2394
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9871 - loss: 0.0456 - val_accuracy: 0.9170 - val_loss: 0.3040
Epoch 5/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9962 - loss: 0.0137 - val_accuracy: 0.9190 - val_loss: 0.3944
Epoch 6/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9990 - loss: 0.0046 - val_accuracy: 0.9170 - val_loss: 0.4221
Epoch 7/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9988 - loss: 0.0030 - val_accuracy: 0.9175 - val_loss: 0.4552
Epoch 8/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9997 - loss: 0.0018 - val_accuracy: 0.

In [42]:
# ANN Model Approach Code Style

In [87]:
from tensorflow.keras.layers import Dense, Input, Embedding, GlobalMaxPooling1D, Flatten
from tensorflow.keras.models import Model

# 1. Create/Convert Seq data into Word Embedding / Embedded Data

# Input layer - It takes in a sequence of integers, i.e. T (time step size / vocab size)
i = Input(shape=(T,))

# Create Word Embedding ---- This layer will take sequence of integers and return a sequence of word vectors
# Embedding dimension D (hyperparameter)
D = 20

# Embedding layer
x = Embedding(V + 1, D)(i)

# Flatten the embedding layer output to convert it to a shape suitable for Dense layers
x = Flatten()(x)

# First Dense Layer
x = Dense(128, activation='relu')(x)

# Second Dense Layer (optional, will adjust the number of layers to tweak)
x = Dense(64, activation='relu')(x)

# Third Dense Layer (optional, will add/remove based on complexity required)
x = Dense(32, activation='relu')(x)

# Output Layer (Sigmoid for binary classification)
x = Dense(1, activation='sigmoid')(x)

# Create Model
model = Model(i, x)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Model Summary
model.summary()

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)           │ (None, 953)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ embedding_9 (Embedding)              │ (None, 953, 20)             │         556,760 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_2 (Flatten)                  │ (None, 19060)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_15 (Dense)                     │ (None, 128)                 │       2,439,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_16 (Dense)                     │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_17 (Dense)                     │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_18 (Dense)                     │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,006,937 (11.47 MB)

 Trainable params: 3,006,937 (11.47 MB)

 Non-trainable params: 0 (0.00 B)

In [88]:
#Fit
history = model.fit(trainData,y_train, epochs=50, validation_data=(testData,y_test))

Epoch 1/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9063 - loss: 0.3373 - val_accuracy: 0.9085 - val_loss: 0.2224
Epoch 2/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9372 - loss: 0.1438 - val_accuracy: 0.9215 - val_loss: 0.2432
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9868 - loss: 0.0405 - val_accuracy: 0.9230 - val_loss: 0.3238
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9977 - loss: 0.0098 - val_accuracy: 0.9175 - val_loss: 0.2927
Epoch 5/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9989 - loss: 0.0034 - val_accuracy: 0.9070 - val_loss: 0.4043
Epoch 6/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9997 - loss: 0.0019 - val_accuracy: 0.8815 - val_loss: 0.5911
Epoch 7/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9996 - loss: 0.0042 - val_accuracy: 0.9105 - val_loss: 0.4448
Epoch 8/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9825 - loss: 0.1181 - val_accuracy: 0.